In [1]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint

In [2]:
class JokeState(TypedDict):

    topic: str
    joke: str
    explanation: str

In [8]:
llm = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-72B-Instruct",
    task="text-generation",
    huggingfacehub_api_token="YOUR_API_TOKEN"
)

model = ChatHuggingFace(llm = llm)

In [9]:
def generate_joke(state: JokeState):

    prompt = f'generate a joke on the topic {state["topic"]}'
    response = model.invoke(prompt).content
    return {'joke': response}

In [10]:
def generate_explanation(state: JokeState):

    prompt = f'write an explanation for the joke - {state["joke"]}'
    response = model.invoke(prompt).content

    return {'explanation': response}

In [11]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [12]:
config1 = {"configurable": {"thread_id": "1"}}
workflow.invoke({'topic':'pizza'}, config=config1)

{'topic': 'pizza',
 'joke': 'Why did the pizza go to the doctor?\n\nBecause it was feeling a little saucy!',
 'explanation': 'The joke "Why did the pizza go to the doctor? Because it was feeling a little saucy!" plays on a double meaning of the word "saucy." \n\n1. **Literal Meaning:** "Saucy" can describe food that has a lot of sauce, which is a common characteristic of pizza. The pizza, in this context, has too much sauce and is therefore "feeling saucy."\n\n2. **Figurative Meaning:** "Saucy" can also mean bold, flirtatious, or a bit mischievous. When applied to a person, it suggests someone who is being a bit cheeky or playful. The joke anthropomorphizes the pizza, giving it human-like feelings and behaviors, and suggests that it\'s acting out of the ordinary, much like a person might feel when they\'re being a bit naughty or playful.\n\nBy combining these two meanings, the joke creates a pun that is both playful and clever. The pizza, feeling overly saucy (full of sauce), goes to t